# T37 - 同花顺Excel导入财务指标

## 3. 数据采集与ETL

本章详细说明数据采集和ETL（抽取、转换、加载）流程。

## 3.1 环境准备

In [ ]:
# 导入标准库
import os
import sys
import time
from datetime import datetime, timedelta
from pathlib import Path
from time import sleep

# 添加项目路径
PROJECT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_DIR))

# 导入第三方库
import pandas as pd
import numpy as np
import sqlalchemy
from sqlalchemy import text
import psycopg2

# 导入项目配置
from config import (
    get_mysql_bond_engine,
    get_postgres_engine,
    get_postgres_connection,
    get_ths_account,
    CHUNK_SIZE,
    SQL_RETRY_ATTEMPTS,
    SQL_RETRY_DELAY,
    THS_FINANCIAL_INDICATORS
)

print("环境准备完成")

## 3.2 数据库连接管理

### 3.2.1 SQL重试机制

In [ ]:
class SQLRetryManager:
    """SQL执行重试管理器"""
    
    def __init__(self, engine, max_retries=SQL_RETRY_ATTEMPTS, retry_delay=SQL_RETRY_DELAY):
        """
        初始化重试管理器
        
        Parameters:
        -----------
        engine : SQLAlchemy Engine
            数据库引擎
        max_retries : int
            最大重试次数
        retry_delay : int
            重试延迟(秒)
        """
        self.engine = engine
        self.max_retries = max_retries
        self.retry_delay = retry_delay
        self.connection = None
        
    def connect(self):
        """建立数据库连接"""
        self.connection = self.engine.connect()
        return self.connection
    
    def reconnect(self):
        """重新建立连接"""
        if self.connection:
            try:
                self.connection.close()
            except:
                pass
        return self.connect()
    
    def execute_with_retry(self, sql, params=None):
        """
        带重试机制的SQL执行
        
        Parameters:
        -----------
        sql : str
            SQL语句
        params : dict
            参数
            
        Returns:
        --------
        ResultProxy : 执行结果
        """
        retry_count = 0
        last_error = None
        
        while retry_count < self.max_retries:
            try:
                if self.connection is None:
                    self.connect()
                    
                trans = self.connection.begin()
                try:
                    if params:
                        result = self.connection.execute(text(sql), params)
                    else:
                        result = self.connection.execute(text(sql))
                    trans.commit()
                    return result
                except Exception as e:
                    trans.rollback()
                    raise e
                    
            except Exception as e:
                last_error = e
                retry_count += 1
                print(f"执行失败 (尝试 {retry_count}/{self.max_retries}): {e}")
                self.reconnect()
                sleep(self.retry_delay)
                
        raise Exception(f"SQL执行失败，已重试{self.max_retries}次: {last_error}")

# 测试重试管理器
mysql_engine = get_mysql_bond_engine()
retry_manager = SQLRetryManager(mysql_engine)
print("SQL重试管理器初始化完成")

### 3.2.2 PostgreSQL重试机制

In [ ]:
class PostgresRetryManager:
    """PostgreSQL执行重试管理器"""
    
    def __init__(self, max_retries=SQL_RETRY_ATTEMPTS, retry_delay=SQL_RETRY_DELAY):
        """
        初始化重试管理器
        
        Parameters:
        -----------
        max_retries : int
            最大重试次数
        retry_delay : int
            重试延迟(秒)
        """
        self.max_retries = max_retries
        self.retry_delay = retry_delay
        self.connection = None
        self.cursor = None
        self._connect()
        
    def _connect(self):
        """建立数据库连接"""
        self.connection = get_postgres_connection()
        self.cursor = self.connection.cursor()
        
    def _reconnect(self):
        """重新建立连接"""
        if self.connection:
            try:
                self.connection.close()
            except:
                pass
        self._connect()
        
    def execute_with_retry(self, sql, params=None):
        """
        带重试机制的SQL执行
        
        Parameters:
        -----------
        sql : str
            SQL语句
        params : tuple
            参数
            
        Returns:
        --------
        cursor : 执行后的游标
        """
        retry_count = 0
        last_error = None
        
        while retry_count < self.max_retries:
            try:
                self.connection.autocommit = False
                if params:
                    self.cursor.execute(sql, params)
                else:
                    self.cursor.execute(sql)
                self.connection.commit()
                return self.cursor
                
            except Exception as e:
                last_error = e
                self.connection.rollback()
                retry_count += 1
                print(f"执行失败 (尝试 {retry_count}/{self.max_retries}): {e}")
                self._reconnect()
                sleep(self.retry_delay)
                
        raise Exception(f"PostgreSQL执行失败，已重试{self.max_retries}次: {last_error}")

# 初始化PostgreSQL重试管理器
pg_retry_manager = PostgresRetryManager()
print("PostgreSQL重试管理器初始化完成")

## 3.3 发行人数据提取

从数据库查询债券发行人代码列表。

In [ ]:
def get_issuer_trade_codes(pg_engine):
    """
    获取发行人代码列表
    
    Parameters:
    -----------
    pg_engine : SQLAlchemy Engine
        PostgreSQL数据库引擎
        
    Returns:
    --------
    list : 发行人代码列表
    """
    sql = """
    SELECT 
        max(trade_code) AS trade_code,
        ths_issuer_name_cn_bond
    FROM (
        SELECT trade_code, ths_issuer_name_cn_bond FROM basicinfo_credit
        UNION ALL
        SELECT trade_code, ths_issuer_name_cn_bond FROM basicinfo_finance
        UNION ALL
        SELECT trade_code, ths_issuer_name_cn_bond FROM basicinfo_abs
    ) SQ
    GROUP BY ths_issuer_name_cn_bond
    """
    
    df = pd.read_sql(sql, pg_engine)
    trade_codes = df['trade_code'].tolist()
    
    print(f"获取到 {len(trade_codes)} 个发行人代码")
    return trade_codes

# 获取发行人代码
pg_engine = get_postgres_engine()
trade_codes = get_issuer_trade_codes(pg_engine)
print(f"\n前5个发行人代码: {trade_codes[:5]}")

## 3.4 同花顺API数据提取

### 3.4.1 API登录

In [ ]:
def login_ths_api():
    """
    登录同花顺API
    
    Returns:
    --------
    bool : 登录是否成功
    """
    try:
        from iFinDPy import THS_iFinDLogin
        
        account = get_ths_account()
        if account is None:
            print("未配置同花顺API账号")
            return False
            
        result = THS_iFinDLogin(account['username'], account['password'])
        
        if result == 0:
            print("同花顺API登录成功")
            return True
        else:
            print(f"同花顺API登录失败，错误码: {result}")
            return False
            
    except ImportError:
        print("iFinDPy模块未安装，请安装同花顺iFinD终端")
        return False
    except Exception as e:
        print(f"登录异常: {e}")
        return False

# 登录同花顺API（需要安装iFinD终端）
# login_success = login_ths_api()

### 3.4.2 财务指标提取

In [ ]:
def build_ths_params(dt, dt1):
    """
    构建同花顺API参数字符串
    
    Parameters:
    -----------
    dt : str
        当前报告期（如20230630）
    dt1 : str
        上期报告期（如2022-12-31）
        
    Returns:
    --------
    str : 参数字符串
    """
    # 简化版参数构建（实际使用时需要根据指标数量调整）
    params = f'{dt},100;{dt};{dt};{dt};{dt};{dt};{dt};{dt};{dt};{dt}'
    params += f';{dt},100;{dt},100;{dt},100;{dt},100;{dt},100'
    params += f';{dt1},100;{dt1},100;{dt1},100;{dt1},100;{dt1},100'
    return params

def fetch_financial_indicators(trade_code, dt, dt1):
    """
    从同花顺API提取财务指标
    
    Parameters:
    -----------
    trade_code : str
        债券代码
    dt : str
        当前报告期
    dt1 : str
        上期报告期
        
    Returns:
    --------
    DataFrame : 财务指标数据
    """
    try:
        from iFinDPy import THS_BD
        
        # 构建指标列表（简化版，实际需要150+个指标）
        indicators = ';'.join(THS_FINANCIAL_INDICATORS[:20])  # 使用前20个作为示例
        params = build_ths_params(dt, dt1)
        
        # 调用API
        df = THS_BD(trade_code, indicators, params)
        
        if df is None or df.data is None:
            return pd.DataFrame()
            
        df_result = df.data
        df_result['dt'] = dt
        
        return df_result
        
    except ImportError:
        print("iFinDPy模块未安装")
        return pd.DataFrame()
    except Exception as e:
        print(f"提取财务指标失败: {e}")
        return pd.DataFrame()

# 示例：提取单个发行人的财务指标
print("财务指标提取函数定义完成")

## 3.5 Excel文件处理

### 3.5.1 Excel读取

In [ ]:
import openpyxl
from openpyxl import load_workbook

def read_financial_excel(file_path):
    """
    读取财务Excel文件
    
    Parameters:
    -----------
    file_path : str
        Excel文件路径
        
    Returns:
    --------
    DataFrame : 财务数据
    """
    try:
        # 使用openpyxl读取
        wb = load_workbook(file_path)
        sheet_names = wb.sheetnames
        print(f"工作表: {sheet_names}")
        
        # 使用pandas读取
        df = pd.read_excel(file_path)
        print(f"读取到 {len(df)} 行数据")
        
        return df
        
    except Exception as e:
        print(f"读取Excel失败: {e}")
        return pd.DataFrame()

print("Excel读取函数定义完成")

### 3.5.2 Excel日期填充

In [ ]:
def extract_date_from_filename(filename):
    """
    从文件名提取日期
    
    Parameters:
    -----------
    filename : str
        文件名（如20220630.xlsx）
        
    Returns:
    --------
    str : 格式化日期（如2022-06-30）
    """
    try:
        date_str = filename.split('.')[0]  # 去除扩展名
        date_obj = datetime.strptime(date_str, '%Y%m%d')
        return date_obj.strftime('%Y-%m-%d')
    except Exception as e:
        print(f"日期提取失败: {e}")
        return None

# 测试日期提取
test_filename = "20220630.xlsx"
formatted_date = extract_date_from_filename(test_filename)
print(f"文件名: {test_filename} -> 日期: {formatted_date}")

### 3.5.3 批量Excel处理

In [ ]:
def batch_process_excel(folder_path, output_folder=None):
    """
    批量处理Excel文件
    
    Parameters:
    -----------
    folder_path : str
        输入文件夹路径
    output_folder : str
        输出文件夹路径（可选）
        
    Returns:
    --------
    list : 处理结果列表
    """
    results = []
    
    # 获取所有Excel文件
    files = [f for f in os.listdir(folder_path) if f.endswith('.xlsx')]
    print(f"找到 {len(files)} 个Excel文件")
    
    for filename in files:
        file_path = os.path.join(folder_path, filename)
        
        try:
            # 读取Excel
            df = read_financial_excel(file_path)
            
            # 提取日期
            date_str = extract_date_from_filename(filename)
            
            result = {
                'filename': filename,
                'rows': len(df),
                'date': date_str,
                'status': 'success'
            }
            
        except Exception as e:
            result = {
                'filename': filename,
                'rows': 0,
                'date': None,
                'status': f'failed: {str(e)}'
            }
            
        results.append(result)
        print(f"处理 {filename}: {result['status']}")
        
    return results

print("批量Excel处理函数定义完成")

## 3.6 数据加载到PostgreSQL

### 3.6.1 批量导入Excel到数据库

In [ ]:
def batch_import_excel_to_postgres(folder_path, table_name, pg_url):
    """
    批量导入Excel文件到PostgreSQL
    
    Parameters:
    -----------
    folder_path : str
        Excel文件夹路径
    table_name : str
        目标表名
    pg_url : str
        PostgreSQL连接URL
        
    Returns:
    --------
    int : 导入的总行数
    """
    total_rows = 0
    
    # 获取所有Excel文件
    files = [f for f in os.listdir(folder_path) 
             if f.endswith('.xlsx') or f.endswith('.xls')]
    
    print(f"找到 {len(files)} 个Excel文件")
    
    for filename in files:
        file_path = os.path.join(folder_path, filename)
        
        try:
            # 读取Excel
            df = pd.read_excel(file_path)
            
            # 导入到PostgreSQL
            df.to_sql(table_name, pg_url, if_exists='append', index=False)
            
            total_rows += len(df)
            print(f"导入 {filename}: {len(df)} 行")
            
        except Exception as e:
            print(f"导入 {filename} 失败: {e}")
            
    print(f"\n总计导入 {total_rows} 行数据")
    return total_rows

print("批量导入函数定义完成")

### 3.6.2 列类型转换

In [ ]:
def convert_column_types(pg_engine, table_name, exclude_columns=['dt', 'thscode']):
    """
    转换表的列类型为FLOAT
    
    Parameters:
    -----------
    pg_engine : SQLAlchemy Engine
        PostgreSQL引擎
    table_name : str
        表名
    exclude_columns : list
        排除的列名
    """
    # 获取所有列名
    sql = f"""
    SELECT column_name 
    FROM information_schema.columns 
    WHERE table_name = '{table_name}'
    """
    df = pd.read_sql(sql, pg_engine)
    columns = df['column_name'].tolist()
    
    # 排除指定列
    columns_to_alter = [col for col in columns if col not in exclude_columns]
    
    print(f"需要转换 {len(columns_to_alter)} 列")
    
    # 使用原生连接执行ALTER
    pg_conn = get_postgres_connection()
    cursor = pg_conn.cursor()
    
    for col in columns_to_alter:
        try:
            cursor.execute(f"""
                ALTER TABLE {table_name}
                ALTER COLUMN {col} TYPE FLOAT USING {col}::FLOAT
            """)
        except Exception as e:
            print(f"转换列 {col} 失败: {e}")
            
    pg_conn.commit()
    cursor.close()
    pg_conn.close()
    
    print("列类型转换完成")

print("列类型转换函数定义完成")

### 3.6.3 宽表转长表

In [ ]:
def wide_to_long_migration(pg_url, source_table, target_table, 
                           id_vars=['dt', 'thscode'], 
                           chunksize=CHUNK_SIZE):
    """
    将宽表数据迁移到长表格式
    
    Parameters:
    -----------
    pg_url : str
        PostgreSQL连接URL
    source_table : str
        源表名（宽表）
    target_table : str
        目标表名（长表）
    id_vars : list
        保持不变的列
    chunksize : int
        分块大小
    """
    query = f"SELECT * FROM {source_table}"
    chunks = pd.read_sql(query, pg_url, chunksize=chunksize)
    
    processed_rows = 0
    
    for df in chunks:
        # 宽表转长表
        df_long = df.melt(
            id_vars=id_vars,
            var_name='指标',
            value_name='value'
        )
        
        # 重命名列
        df_long.rename(columns={'thscode': 'trade_code'}, inplace=True)
        
        # 写入目标表
        try:
            df_long.to_sql(target_table, pg_url, if_exists='append', index=False)
        except Exception as e:
            print(f"写入失败: {e}")
            
        processed_rows += len(df)
        print(f"已处理 {processed_rows} 行")
        
    print(f"\n迁移完成，共处理 {processed_rows} 行")

print("宽表转长表函数定义完成")